In [31]:
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig
import copy
import numpy as np
import biotite.structure.io as bsio
from Bio import Align



In [12]:
## from https://charmm-gui.org/?doc=lecture&module=scientific&lesson=11

#!/usr/bin/env python



def fit_rms(ref_c,c):
    # move geometric center to the origin
    ref_trans = np.average(ref_c, axis=0)
    ref_c = ref_c - ref_trans
    c_trans = np.average(c, axis=0)
    c = c - c_trans

    # covariance matrix
    C = np.dot(c.T, ref_c)

    # Singular Value Decomposition
    (r1, s, r2) = np.linalg.svd(C)

    # compute sign (remove mirroring)
    if np.linalg.det(C) < 0:
        r2[2,:] *= -1.0
    U = np.dot(r1, r2)
    return (c_trans, U, ref_trans)


def set_rmsd(c1, c2):
    rmsd = 0.0
    c_trans, U, ref_trans = fit_rms(c1, c2)
    new_c2 = np.dot(c2 - c_trans, U) + ref_trans
    rmsd = np.sqrt( np.average( np.sum( ( c1 - new_c2 )**2, axis=1 ) ) )
    return rmsd

def get_aligned_coord(self, atoms, name=None):
    new_c2 = copy.deepcopy(atoms)
    for atom in new_c2:
        atom.x, atom.y, atom.z = np.dot(np.array([atom.x, atom.y, atom.z]) - self.c_trans, self.U) + self.ref_trans
    return new_c2

# if __name__ == '__main__':

#     pdbf1 = '1y3q.pdb'
#     pdbf2 = '1y3n.pdb'
    
#     struct1 = bsio.load_structure(pdbf1, extra_fields=["b_factor"])
#     struct2 = bsio.load_structure(pdbf1, extra_fields=["b_factor"])

#     atoms1 = pdbf1.get_atoms(to_dict=False)
#     atoms2 = pdbf2.get_atoms(to_dict=False)

#     rmsd = set_rmsd(struct1.coord, struct2.coord)
#     new_atoms = get_aligned_coord(atoms2)


In [ ]:
import torch

AA_3_letters = ["ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS","ILE","LEU","LYS","MET","PHE","PRO","SER","THR","TRP","TYR","VAL"]
AA_1_letter = list("ARNDCQEGHILKMFPSTWYV")

AA_mapping = {k:v for k,v in zip(AA_3_letters,AA_1_letter)}


def calc_plddt_and_rmsd_score(cur_seq,ref_pdb_path,cur_pdb_path, model):

    ref_struct = bsio.load_structure(ref_pdb_path, extra_fields=["b_factor"])

    ref_atoms = []
    for atom in ref_struct:
        if atom.atom_name == "CA":
            ref_atoms.append(atom.coord)
    ref_atoms =  np.array(ref_atoms)

    aligner = Align.PairwiseAligner()

    ref_seq = set([(int(ind),AA_mapping[str(res)]) for ind,res in zip(ref_struct.res_id, ref_struct.res_name) if str(res) in list(AA_mapping.keys())])
    ref_seq = [seq[1] for seq in ref_seq]
    ref_seq = ''.join(ref_seq)

    alignments = aligner.align(ref_seq, cur_seq)
    alignment = alignments[0].aligned

    aligned_seq1 = ''
    aligned_seq2 = ''

    for i in range(alignment.shape[1]):

        current_range_1 = alignment[0,i,:]
        aligned_seq1 += ref_seq[current_range_1[0]:current_range_1[1]]

        current_range_2 = alignment[1,i,:]
        aligned_seq2 += cur_seq[current_range_2[0]:current_range_2[1]]


    with torch.no_grad():
            pdb_file = model.infer_pdb(aligned_seq2)

    with open(cur_pdb_path, "w") as f:
        f.write(pdb_file)

    struct_cur = bsio.load_structure(cur_pdb_path, extra_fields=["b_factor"])

    cur_atoms = []
    for atom in struct_cur:
        if atom.atom_name == "CA":
            cur_atoms.append(atom.coord)
    cur_atoms =  np.array(cur_atoms)

    rmsd = set_rmsd(ref_atoms, cur_atoms)






